In [1]:
!python -m ensurepip --upgrade
!python -m pip install pandas numpy openpyxl matplotlib

Looking in links: /tmp/tmp63gnbhm0

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [4]:
import pandas as pd
import sys

# ---- Hard-coded file paths ----
FILE_A = "/mnt/projects/mammaprint/data_mapping.csv"
FILE_B = "/mnt/projects/mammaprint/data_mapping_new.csv"

# Output for rows in B but not in A (full rows, not just keys)
ONLY_IN_B = "/mnt/projects/mammaprint/data_mapping_diff2.csv"

KEY = "record_num"


def main():
    # Load CSVs
    df_a = pd.read_csv(FILE_A)
    df_b = pd.read_csv(FILE_B)

    # Ensure key column exists
    if KEY not in df_a.columns or KEY not in df_b.columns:
        print(f"ERROR: '{KEY}' column is missing in one of the files.")
        sys.exit(1)

    # ---- Compare keys ----
    keys_a = df_a[KEY].drop_duplicates()
    keys_b = df_b[KEY].drop_duplicates()

    # Keys in A not found in B
    a_not_in_b = keys_a[~keys_a.isin(keys_b)]

    # ---- FULL ROWS from B that do not exist in A (keep all columns) ----
    b_not_in_a_full = df_b[~df_b[KEY].isin(keys_a)]
    b_not_in_a_full.to_csv(ONLY_IN_B, index=False)

    print(f"Saved {len(b_not_in_a_full)} full rows (B minus A on '{KEY}') to {ONLY_IN_B}")

    # Check subset condition
    if not a_not_in_b.empty:
        print(f"ERROR: A is NOT a subset of B based on key '{KEY}'.")
        print(f"Keys in A but missing in B: {len(a_not_in_b)}")
        print(a_not_in_b)
        sys.exit(1)

    print(f"SUCCESS: A is a subset of B based on key '{KEY}'.")


if __name__ == "__main__":
    main()

Saved 160 full rows (B minus A on 'record_num') to /mnt/projects/mammaprint/data_mapping_diff2.csv
SUCCESS: A is a subset of B based on key 'record_num'.
